# Churn Prediction - Interview Ready Notebook

This notebook rebuilds the churn model in a clean and explainable way, based on the existing project logic and dataset.
It is designed to be easy to present in interviews and easy to connect with FastAPI backend input/output.

## 1. Problem Statement

Customer churn prediction helps identify customers who are likely to leave a service.
By predicting churn early, the business can take action (offers, support, retention campaigns) to reduce revenue loss.

## Existing Project Logic Analysis (Before Rebuild)

From the current codebase:
- Existing production model artifact is a `Pipeline` with:
  - `preprocessor = ColumnTransformer`
  - `model = XGBClassifier`
- Existing preprocessing uses:
  - median imputation + standard scaling for numeric columns
  - most-frequent imputation + one-hot encoding for categorical columns
- Existing backend inference contract expects core JSON features: `age`, `tenure`, `monthly_charges`.

For interview clarity and backend compatibility, this notebook trains a clean **Random Forest** model using those core input fields.
This keeps the pipeline simple while preserving practical production behavior.

In [ ]:
# 2. Import Libraries
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 4)

In [ ]:
# Helper: resolve project root robustly
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'processed' / 'telco_churn_cleaned.csv').exists():
            return candidate
        if (candidate / 'churn-ai-platform' / 'data' / 'processed' / 'telco_churn_cleaned.csv').exists():
            return candidate / 'churn-ai-platform'
    raise FileNotFoundError('Could not find project root with telco_churn_cleaned.csv')

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'telco_churn_cleaned.csv'
MODEL_DIR = PROJECT_ROOT / 'ml' / 'model'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, DATA_PATH

## 3. Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = [str(c).strip().lower() for c in df.columns]

print('Dataset path:', DATA_PATH)
print('Rows loaded:', len(df))
df.head()

## 4. Basic Data Understanding

In this step we inspect dataset size, columns, data types, and target variable.
This helps us confirm what the model will learn from and what we must clean first.

In [ ]:
print('Shape:', df.shape)
print('
Columns:')
print(df.columns.tolist())
print('
Data types:')
display(df.dtypes.to_frame('dtype').head(20))

target_col = 'churn_label' if 'churn_label' in df.columns else 'churn'
print('
Target column selected:', target_col)
print(df[target_col].value_counts(dropna=False))

## 5. Data Cleaning

Why this step is needed:
- Missing values can break model training or reduce quality.
- Duplicate rows can bias model learning.
- Leakage columns (like churn score or churn category) can make model evaluation unrealistic.

In [ ]:
before_rows = len(df)
duplicate_count = df.duplicated().sum()
missing_counts = df.isna().sum().sort_values(ascending=False)

print('Duplicate rows:', int(duplicate_count))
print('Top missing-value columns:')
display(missing_counts.head(10).to_frame('missing_count'))

df = df.drop_duplicates().copy()
after_rows = len(df)
print(f'Rows before: {before_rows}, rows after duplicate removal: {after_rows}')

leakage_cols = [c for c in ['churn_score', 'churn_category', 'customer_status'] if c in df.columns]
print('Leakage columns removed:', leakage_cols)

## 6. Exploratory Data Analysis (EDA)

We check churn distribution and key feature patterns (`tenure`, `monthly_charges`).
This gives intuition about class balance and customer behavior before modeling.

In [ ]:
# Normalize target to binary for plotting and training
target_raw = df[target_col].astype(str).str.strip().str.lower()
y = target_raw.map({'yes': 1, 'no': 0, '1': 1, '0': 0})
if y.isna().any():
    unknown = sorted(target_raw[y.isna()].unique().tolist())
    raise ValueError(f'Unexpected target labels: {unknown}')

plot_df = df.copy()
plot_df['churn_binary'] = y.astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.countplot(x='churn_binary', data=plot_df, ax=axes[0])
axes[0].set_title('Churn Distribution (0=No, 1=Yes)')

tenure_col = 'tenure' if 'tenure' in plot_df.columns else 'tenure_in_months'
charges_col = 'monthly_charges' if 'monthly_charges' in plot_df.columns else 'monthly_charge'

sns.boxplot(x='churn_binary', y=tenure_col, data=plot_df, ax=axes[1])
axes[1].set_title(f'{tenure_col} vs Churn')

sns.boxplot(x='churn_binary', y=charges_col, data=plot_df, ax=axes[2])
axes[2].set_title(f'{charges_col} vs Churn')

plt.tight_layout()
plt.show()

**Quick insights:**
- Class distribution shows churn vs non-churn balance.
- Tenure and monthly charges often separate churn behavior and are useful for baseline modeling.

## 7. Feature Engineering

Why encoding is required:
- ML models need numeric inputs.
- Categorical text fields must be encoded (for example with one-hot encoding).

For backend alignment, we use core API-friendly features:
- `age`
- `tenure`
- `monthly_charges`

We still keep a general preprocessing design that can handle categorical columns if they are added later.

In [ ]:
feature_map = {
    'tenure': 'tenure' if 'tenure' in df.columns else 'tenure_in_months',
    'monthly_charges': 'monthly_charges' if 'monthly_charges' in df.columns else 'monthly_charge',
    'age': 'age'
}

required_cols = [feature_map['age'], feature_map['tenure'], feature_map['monthly_charges']]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f'Missing required model columns: {missing_required}')

X = df[required_cols].copy()
X = X.rename(columns={
    feature_map['age']: 'age',
    feature_map['tenure']: 'tenure',
    feature_map['monthly_charges']: 'monthly_charges'
})

for c in ['age', 'tenure', 'monthly_charges']:
    X[c] = pd.to_numeric(X[c], errors='coerce')

numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

print('Training feature columns:', X.columns.tolist())
print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)

## 8. Train-Test Split

Purpose:
- Train set teaches the model patterns.
- Test set checks performance on unseen data.

We use an 80-20 split with stratification to preserve churn class ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y.astype(int),
    test_size=0.2,
    random_state=42,
    stratify=y.astype(int)
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

## 9. Model Training

Based on existing project training options, we use **Random Forest** here for interview-friendly clarity.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf_model)
])

pipeline.fit(X_train, y_train)
print('Model training complete.')

## 10. Model Evaluation

- **Accuracy**: percentage of correct predictions.
- **Confusion Matrix**: shows correct and incorrect predictions by class.
  - Top-left: true non-churn
  - Bottom-right: true churn
  - Off-diagonal: errors

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f'Accuracy: {acc:.4f}')
print('\nClassification Report:\n')
print(report)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 11. Prediction Example

Here we predict churn probability for one sample.
The output probability is between 0 and 1, which is API-friendly.

In [ ]:
sample_input = pd.DataFrame([
    {
        'age': 45,
        'tenure': 10,
        'monthly_charges': 95.0
    }
])

sample_prob = float(pipeline.predict_proba(sample_input)[0, 1])
sample_pred = int(sample_prob >= 0.5)

print('Sample input:', sample_input.to_dict(orient='records')[0])
print(f'Predicted churn probability: {sample_prob:.4f}')
print('Predicted class (1=Churn, 0=No Churn):', sample_pred)

## 12. Save Model and Artifacts

We save:
- model pipeline (`churn_model.pkl`)
- preprocessor (`preprocessor.joblib`)
- API feature list (`feature_names.joblib`)

In [ ]:
model_path = MODEL_DIR / 'churn_model.pkl'
preprocessor_path = MODEL_DIR / 'preprocessor.joblib'
feature_names_path = MODEL_DIR / 'feature_names.joblib'

joblib.dump(pipeline, model_path)
joblib.dump(preprocessor, preprocessor_path)
joblib.dump(X.columns.tolist(), feature_names_path)

print('Saved model to:', model_path)
print('Saved preprocessor to:', preprocessor_path)
print('Saved feature names to:', feature_names_path)

## API Compatibility Helper (FastAPI Friendly)

This helper ensures JSON keys align with backend-style input and returns probability output.

In [ ]:
def predict_from_json(payload: dict, trained_pipeline: Pipeline) -> dict:
    features = payload.get('features', payload)
    sample = {
        'age': features.get('age'),
        'tenure': features.get('tenure', features.get('tenure_in_months')),
        'monthly_charges': features.get('monthly_charges', features.get('monthly_charge'))
    }

    frame = pd.DataFrame([sample])
    for col in ['age', 'tenure', 'monthly_charges']:
        frame[col] = pd.to_numeric(frame[col], errors='coerce')

    probability = float(trained_pipeline.predict_proba(frame)[0, 1])
    prediction = int(probability >= 0.5)
    risk = 'Low' if probability < 0.3 else ('Medium' if probability < 0.7 else 'High')

    return {
        'prediction': prediction,
        'probability': probability,
        'risk_level': risk
    }

api_test_payload = {
    'customer_id': 'CUST-1001',
    'features': {
        'age': 39,
        'tenure': 7,
        'monthly_charges': 89.5
    }
}

predict_from_json(api_test_payload, pipeline)

## 13. Final Conclusion

- We built a clear churn prediction pipeline from data loading to model saving.
- We used a clean, explainable Random Forest setup that is easy to discuss in interviews.
- The model returns churn probability (0-1) and supports backend-friendly JSON input fields.
- This notebook is ready for demonstration, iteration, and FastAPI integration.